# 10 · Phase 2: Legal NER on top of the Phase-1 models
Flow: download the Phase-2 data, check offsets/alignment with **your** tokenizer, pick a Phase-1 run + an adaptation
method, train (val = 10 held-out docs, strict macro-F1 every epoch, best epoch kept), then predict fold 3 →
`submission.zip`. Each run goes to `MyDrive/cs781/ner_runs/<run_name>/` (config, log, best.pt, val predictions,
results.json with scores **and** trainable parameters for the adaptation comparison).

Brief (§8.1–8.2): use **all four** Phase-1 models; ≥4 methodologically different experiments; **LoRMA compulsory**;
**1 knowledge-distillation** run (larger teacher → smaller student; `kd_teacher=`). Your methods go in `adapters/`.

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install ijson "huggingface_hub[hf_xet]" tokenizers
import os
from huggingface_hub import login, whoami

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    p = f'{CODE_DIR}/hf_token.txt'
    if os.path.exists(p):
        return open(p).read().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(f'No HF token found: set $HF_TOKEN or create {p}') from e

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in to Hugging Face as', whoami(token=HF_TOKEN)['name'])

In [ ]:
# Phase-2 data -> /content/p2 (and a copy on Drive)
from huggingface_hub import hf_hub_download
import shutil
P2 = '/content/p2'; os.makedirs(P2, exist_ok=True); os.makedirs(f'{DRIVE_ROOT}/data/phase2', exist_ok=True)
for f in ['Train/fold1-text.json', 'Train/fold1-gold.json', 'Train/fold2-text.json', 'Train/fold2-gold.json',
          'Train/labels.txt', 'Test/fold3-text.json']:
    p = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_2_LNER/{f}', repo_type='dataset', token=HF_TOKEN)
    shutil.copyfile(p, f'{P2}/{os.path.basename(f)}')
    shutil.copyfile(p, f'{DRIVE_ROOT}/data/phase2/{os.path.basename(f)}')
!ls -lh {P2}

In [ ]:
# Phase-1 runs available as backbones (all four architectures are required this phase)
PHASE1 = {
    'A': f'{RUNS_DIR}/A_qwen_dense_full',   # or A_qwen_dense (10% run)
    'B': f'{RUNS_DIR}/B_encdec',
    'C': f'{RUNS_DIR}/C_gdn_hybrid',
    'D': f'{RUNS_DIR}/D_mla_moe',
}
for k, v in PHASE1.items():
    print(k, v, '| ckpts:', sorted(f for f in os.listdir(v) if f.endswith('.pt')) if os.path.isdir(v) else 'MISSING')
if os.path.isdir(PHASE1['C']):
    !pip -q install flash-linear-attention

In [ ]:
# data checks with YOUR tokenizer: offsets, alignment, and the oracle ceiling (gold -> BIO -> spans)
from tokenizers import Tokenizer
from lm_lib.ner import data as nd, metrics as nm
labels, tags, tag2id = nd.load_labels(f'{P2}/labels.txt')
texts, golds = nd.load_folds(P2, [1, 2])
tok = Tokenizer.from_file(f"{PHASE1['A']}/tokenizer.json")
print(nd.check_gold(texts, golds))
ds = nd.NERDataset(texts, golds, tok, tag2id, window=1024, stride=768)
lens = [len(d['ids']) for d in ds.docs.values()]
print(f'{len(texts)} docs | tokens/doc mean {sum(lens)/len(lens):.0f}, max {max(lens)} | windows {len(ds.examples)} | alignment {ds.align_stats}')
oracle = {d: nd.bio_to_spans(doc['labels'], doc['spans'], texts[d], tags) for d, doc in ds.docs.items()}
print('ORACLE ceiling for token-level BIO:\n' + nm.format_scores(nm.strict_scores(oracle, golds, labels)))

In [ ]:
from lm_lib.ner.trainer import NERConfig, train_ner, predict_test
cfg = NERConfig(
    phase1_run=PHASE1['A'], phase1_ckpt='best',
    method='full',                 # full | linear_probe | partial | lorma | lora | <your adapters/NAME.py>
    method_kwargs={},              # e.g. {'k': 4} for partial, your LoRMA rank/targets, ...
    data_dir=P2, out_dir=f'{DRIVE_ROOT}/ner_runs',
    run_name='',                   # default: <phase1 run>__<method>
    window=1024, stride=768,       # B (encoder trained on 768 tokens): consider window=768, stride=512
    n_val_docs=10, batch_size=8, epochs=10,
    lr=5e-5, head_lr=1e-3, weight_decay=0.01, warmup_frac=0.1, dropout=0.1,
    class_weighting='none',        # or 'inv_sqrt'
    kd_teacher='',                 # KD: path of a trained teacher run in ner_runs/, plus kd_alpha / kd_temperature
)
print(cfg)

In [ ]:
results = train_ner(cfg)

In [ ]:
# span-decoding strategies on this run's held-out val docs (one forward pass, no retraining):
#   decode: argmax (independent per token) | viterbi (best *valid* BIO sequence)
#   window_agg: owner (window where the token has most context) | mean (average over overlapping windows)
from lm_lib.ner.trainer import eval_decoding
run_dir = f'{DRIVE_ROOT}/ner_runs/{cfg.run_name}'
dec_rows = eval_decoding(run_dir)

In [ ]:
# predict fold 3 with the best epoch + YOUR chosen decoding -> prediction.json + submission.zip
DECODE, WINDOW_AGG = 'argmax', 'owner'      # <- your choice from the comparison above
zip_path, test_preds = predict_test(run_dir, decode=DECODE, window_agg=WINDOW_AGG)
from google.colab import files
files.download(zip_path)

In [ ]:
# all Phase-2 runs side by side (for the report)
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in glob.glob(f'{DRIVE_ROOT}/ner_runs/*/results.json')]
df = pd.DataFrame(rows)[['run', 'phase1_run', 'method', 'kd_teacher', 'best_val_macro_f1', 'trainable_params',
                         'total_params', 'trainable_pct']]
df.sort_values('best_val_macro_f1', ascending=False)